# 02 — Compute features on t1 and t2

Applies the feature functions in `src/features.py` to both sides of every revision pair. Uses spaCy `en_core_web_lg` word vectors and Empath seed word lists for category-similarity scoring.

In [1]:
import os, sys
import pandas as pd
import spacy
sys.path.insert(0, os.path.dirname(os.path.abspath('.')) + '/src' if os.path.basename(os.getcwd())!='src' else '.')
import features as F

nlp = spacy.load('en_core_web_lg')
seeds = F.load_seed_words()
centroids = F.build_centroids(nlp, seeds)
thresholds = F.build_thresholds(nlp, seeds, centroids)
print('categories:', list(centroids.keys()))
print('thresholds:', {k: round(v, 3) for k, v in thresholds.items()})

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data') if os.path.basename(os.getcwd()) == 'src' else 'data'
pairs = pd.read_parquet(os.path.join(DATA_DIR, 'revision_pairs.parquet'))
print('pairs', pairs.shape)

categories: ['critical_thinking', 'confusion']
thresholds: {'critical_thinking': 0.656, 'confusion': 0.523}
pairs (267, 20)


In [2]:
def _expand(text, suffix):
    feats = F.compute_all(text, nlp, centroids, thresholds)
    return {f'{k}_{suffix}': v for k, v in feats.items()}

feat_t1 = pairs['text_t1'].apply(lambda t: _expand(t, 't1')).apply(pd.Series)
feat_t2 = pairs['text_t2'].apply(lambda t: _expand(t, 't2')).apply(pd.Series)
pairs_feat = pd.concat([pairs, feat_t1, feat_t2], axis=1)

# pairwise edit distance
pairs_feat['levenshtein'] = [F.levenshtein_distance(a, b) for a, b in zip(pairs['text_t1'], pairs['text_t2'])]
pairs_feat['normalized_levenshtein'] = [F.normalized_levenshtein(a, b) for a, b in zip(pairs['text_t1'], pairs['text_t2'])]

# deltas for each feature
for col in F.FEATURE_COLUMNS + ['content_score','language_score','similarity_score','containment_score']:
    c1, c2 = f'{col}_t1', f'{col}_t2'
    if c1 in pairs_feat.columns and c2 in pairs_feat.columns:
        pairs_feat[f'{col}_delta'] = pairs_feat[c2] - pairs_feat[c1]

print('feature columns added')
print('shape', pairs_feat.shape)

feature columns added
shape (267, 41)


In [3]:
# spot check
cols_check = ['condition','word_count_t1','word_count_t2',
              'critical_thinking_t1','critical_thinking_t2',
              'levenshtein','content_score_t1','content_score_t2']
pairs_feat[cols_check].head(10)

,condition,word_count_t1,word_count_t2,critical_thinking_t1,critical_thinking_t2,levenshtein,content_score_t1,content_score_t2
0,stairs,70.0,100.0,0.000000,0.000000,211,-0.797054,-0.236283
1,stairs,56.0,97.0,0.000000,0.000000,323,-0.093934,0.578476
2,stairs,59.0,55.0,0.000000,0.000000,156,-0.779259,-0.209860
3,stairs,72.0,93.0,1.388889,1.075269,125,-0.258607,0.148138
4,random_reread,51.0,51.0,0.000000,0.000000,0,-0.309791,-0.309791
5,random_reread,56.0,56.0,0.000000,0.000000,0,-0.201603,-0.201603
6,random_reread,60.0,60.0,3.333333,3.333333,0,0.020150,0.020150
7,stairs,63.0,144.0,0.000000,0.000000,552,-0.314404,0.019152
8,stairs,192.0,100.0,0.520833,0.000000,954,0.922634,-0.360719
9,random_reread,59.0,59.0,1.694915,1.694915,0,-0.815462,-0.815462


In [4]:
out_path = os.path.join(DATA_DIR, 'revision_pairs_features.parquet')
pairs_feat.to_parquet(out_path, index=False)
print('wrote', out_path, pairs_feat.shape)

wrote /home/jovyan/active-projects/itell-critical-thinking-analysis/data/revision_pairs_features.parquet (267, 41)
